# **Семинар 5: Нейронные сети - от задания архитектуры до построения прогноза на примере библиотеки TensorFlow**

## Содержание занятия:

### Тема 1. Повторение основных моментов

### Тема 2. Градиентный спуск - продолжение

### Тема 3. TensorFlow: архитектура модели и её компиляция

### Тема 4. TensorFlow: обучение модели

### Тема 5. TensorFlow: инференс (построение прогноза) модели

In [ ]:
# SymPy — это библиотека Python для выполнения символьных вычислений
import sympy as sp
import matplotlib.pyplot as plt
import numpy as np

import numpy as np
from sklearn.datasets import make_regression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Тема 1. Повторение основных моментов (из прошлого семинара)

Нейронные сети — это вычислительные модели, которые состоят из взаимосвязанных узлов (нейронов), организованных слоями.

### Основные компоненты нейронной сети

*   **Входной слой:** Получает исходные данные (например, векторизованный текст или эмбеддинги).
*   **Скрытые слои:** Промежуточные слои, где происходят основные вычисления. В глубоких нейронных сетях таких слоев несколько.
*   **Выходной слой:** Формирует конечный результат (например, вероятность принадлежности текста к определенному классу).
*   **Нейроны:** Базовые вычислительные единицы, которые получают входные сигналы, применяют к ним взвешенную сумму и передают результат через функцию активации.
*   **Веса и смещения:** Параметры модели, которые подстраиваются в процессе обучения. Веса определяют силу связи между нейронами, а смещения сдвигают выходное значение нейрона.

### Функции активации

Функция активации определяет выход нейрона на основе его входных данных. Она вносит нелинейность в модель, что позволяет нейронным сетям обучаться сложным зависимостям в данных. Некоторые распространенные функции активации:

*   **Сигмоида (Sigmoid):** Сжимает входное значение в диапазон от 0 до 1. Часто используется в выходных слоях для задач бинарной классификации.
*   **Гиперболический тангенс (Tanh):** Сжимает входное значение в диапазон от -1 до 1. Похожа на сигмоиду, но центрирована около нуля.
*   **ReLU (Rectified Linear Unit):** Возвращает входное значение, если оно положительное, и 0 в противном случае (max(0, x)). Является одной из самых популярных функций активации в скрытых слоях благодаря своей простоте и способности решать проблему "затухающих градиентов".
*   **Softmax:** Преобразует вектор входных значений в распределение вероятностей. Обычно используется в выходном слое для задач многоклассовой классификации.

### Функции потерь (Loss Functions)

Функция потерь измеряет, насколько хорошо модель справляется с задачей, сравнивая предсказанные значения с фактическими. Цель обучения нейронной сети — минимизировать эту функцию. Примеры функций потерь:

*   **Mean Squared Error (MSE):** Используется для задач регрессии.
*   **Cross-Entropy (Кросс-энтропия):** Используется для задач классификации. `Binary Cross-Entropy` для бинарной классификации и `Categorical Cross-Entropy` для многоклассовой.

# Тема 2. Градиентный спуск - продолжение

**Градиентный спуск** — это основной алгоритм оптимизации, используемый для обучения нейронных сетей. Он итеративно корректирует веса и смещения модели в направлении, противоположном градиенту функции потерь (направлению наибольшего возрастания функции). Это позволяет постепенно уменьшать ошибку модели.

**Скорость обучения (Learning Rate):** Параметр, определяющий размер шага при корректировке весов во время градиентного спуска. Слишком большая скорость может привести к перескакиванию минимума, а слишком маленькая — к очень медленному обучению.

**Проблемы стандартного градиентного спуска:**
*   Может застрять в локальных минимумах
*   Медленно сходится на плоских участках функции потерь
*   Требует вычисления градиента по всему набору данных на каждой итерации (Batch Gradient Descent), что может быть вычислительно затратно для больших данных.

**Модификации градиентного спуска:**

*   **Stochastic Gradient Descent (SGD):** На каждой итерации обновляет веса на основе градиента, вычисленного по одному случайно выбранному примеру данных. Это делает процесс обучения более быстрым и позволяет избежать застревания в некоторых локальных минимумах, но обновления весов при этом содержат больше шума.
*   **Mini-Batch Gradient Descent:** Компромисс между Batch GD (стандартный градиентный спуск) и SGD. Использует мини-батчи (небольшие случайные подмножества данных) для вычисления градиента и обновления весов. Наиболее часто используемый метод на практике.
*   **Momentum:** Добавляет "импульс" к обновлению весов, помогая алгоритму преодолевать локальные минимумы и быстрее сходиться, особенно на плоских участках. Обновление весов зависит не только от текущего градиента, но и от градиентов на предыдущих шагах.
*   **Adagrad (Adaptive Gradient):** Адаптирует скорость обучения для каждого параметра модели индивидуально, уменьшая скорость для часто обновляемых параметров и увеличивая для редких. Может сильно уменьшать скорость обучения со временем.
*   **RMSprop (Root Mean Square Propagation):** Похож на Adagrad, но использует скользящее среднее квадратов градиентов, что помогает избежать чрезмерного уменьшения скорости обучения.
*   **Adam (Adaptive Moment Estimation):** Объединяет идеи Momentum и RMSprop. Адаптирует скорость обучения для каждого параметра и использует экспоненциально затухающие средние градиентов и их квадратов. Один из самых популярных и эффективных оптимизаторов на сегодняшний день.



В прошлый раз мы посмотрели, как работает простейшая реализация метода градиентного спуска для функции одной переменной. Теперь попробуем модифицировать этот метод.

# Задание 1: Модифицированный градиентный спуск (Momentum):

Реализуйте модификацию метода градиентного спуска - Momentum, и с помощью этого метода найдите глобальный минимум функции $f(x) = (x+5)*(x+2)*(x-3)*(x-4)$

Метод задаётся следующей схемой:

$$x_{new} = x_{old} - h_{new},$$
$$h_{new} = \alpha\cdot h_{old} + \eta\cdot f'(x_{old})$$

В коде ниже
* $\eta$ обозначена как `learning_rate`
* $\alpha$ обозначена как `momentum`.

Функия 'momentum_method()' должна возвращать координату точки (x_curr), в которой функция достигает минимума, "след" процесса градиентного спуска (trace) для дальнейшей визуализации

In [ ]:
# Функция создаёт символьные переменные для использования в математических выражениях
x = sp.Symbol('x')

In [ ]:
def f(x):
    return (x + 5) * (x + 2) * (x - 3) * (x - 4)

In [ ]:
def momentum_method(x_start, learning_rate, num_iterations, momentum, delta=0.01):

    x_curr = x_start
    # Производная f(x)
    df_x = sp.diff(f(x))

    trace = []
    trace.append(x_curr)

    h_curr = 0
    h_trace = []
    h_trace.append(h_curr)

    for i in range(1, num_iterations):

        h_new = # ваш код здесь
        x_new = # ваш код здесь

        trace.append(x_new)
        h_trace.append(h_new)


        # Критерий остановки, когда градиент близок к 0
        if abs(df_x.subs(x, x_new)) < delta:
            return # ваш код здесь

        x_curr = x_new
        h_curr = h_new


    return # ваш код здесь

Проверьте, решает ли Momentum проблему застревания в локальном минимуме?

Используйте следующие гиперпараметры
* `x_start = 7`
* `momentum = 0.2`
* `learning_rate=0.001`
* `num_iterations=100`
* `delta=1e-3`

In [ ]:
xmin, trace, num_iter = # ваш код здесь

x_values = [x for x in np.arange(-5, 6, 0.1)]
f_values = [f(x) for x in x_values]

plt.figure(figsize=(10,10))

plt.axvline(x=0, c = 'black')
plt.axhline(y=0, c = 'black')

plt.plot(x_values, f_values)

plt.xlim([-5, 6])
plt.ylim([-120, 150])

plt.title('График функции f(x) c экстремумами и точками перегиба')

plt.xlabel('x')
plt.ylabel('f(x)')

trace_values = [f(x) for x in trace]
plt.scatter(trace, trace_values, c='red')
plt.scatter([xmin],[f(xmin)], c='green')

plt.show()

Теперь запустите метод с гиперпараметрами:

* `x_start = 7`
* `momentum = 0.8`.

Остальные значения гиперпараметров оставьте как и раньше.

In [ ]:
xmin, trace, num_iter = # ваш код здесь

x_values = [x for x in np.arange(-5, 6, 0.1)]
f_values = [f(x) for x in x_values]

plt.figure(figsize=(10,10))

plt.axvline(x=0, c = 'black')
plt.axhline(y=0, c = 'black')

plt.plot(x_values, f_values)

plt.xlim([-5, 6])
plt.ylim([-120, 150])

plt.title('График функции f(x) c экстремумами и точками перегиба')

plt.xlabel('x')
plt.ylabel('f(x)')

trace_values = [f(x) for x in trace]
plt.scatter(trace, trace_values, c='red')
plt.scatter([xmin],[f(xmin)], c='green')

plt.show()

# Тема 3. TensorFlow: архитектура модели и её компиляция

### Генерация данных

Создадим синтетический набор данных для задачи регрессии, используя `sklearn.datasets`, и разделим его на обучающую и тестовую выборки.

In [ ]:
# Синтетический набор данных для задачи регрессии
X, y = make_regression(n_samples=1000, n_features=10, n_informative=5, noise=10, random_state=42)

# Делим набор данных на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Форма X_train:", X_train.shape)
print("Форма X_test:", X_test.shape)
print("Форма y_train:", y_train.shape)
print("Форма y_test:", y_test.shape)

### Что такое TensorFlow и Keras?

### TensorFlow

**TensorFlow** - это библиотека с открытым исходным кодом для численных вычислений, особенно хорошо подходящая для крупномасштабного машинного обучения. Разработана Google и широко используется для создания и обучения нейронных сетей.

Основные возможности TensorFlow:

*   **Многомерные массивы (тензоры):** Основной единицей данных в TensorFlow является тензор, многомерный массив, который позволяет эффективно работать с большими объемами данных.
*   **Автоматическое дифференцирование:** TensorFlow автоматически вычисляет градиенты, что критически важно для алгоритмов оптимизации, таких как градиентный спуск, используемый при обучении нейронных сетей.
*   **Гибкая архитектура:** Поддерживает развертывание на различных платформах, включая CPU, GPU, TPU, а также на мобильных и периферийных устройствах.
*   **Распределенные вычисления:** Позволяет распределять обучение моделей на множество устройств и серверов, ускоряя процесс для очень больших моделей и наборов данных.
*   **Экосистема инструментов:** Включает инструменты, такие как TensorBoard для визуализации процесса обучения, TensorFlow Serving для развертывания моделей и TensorFlow Lite для мобильных и встраиваемых систем.

### Keras

**Keras** - это высокоуровневый API для построения и обучения моделей машинного обучения. Он был разработан с акцентом на быструю разработку, экспериментирование и простоту использования. Изначально Keras был отдельным проектом, но теперь он интегрирован в TensorFlow как `tf.keras` и является рекомендуемым способом работы с нейронными сетями в TensorFlow.

Основные преимущества Keras:

*   **Простота и удобство:** Позволяет быстро и легко создавать сложные нейронные сети благодаря понятному и модульному API.
*   **Гибкость:** Поддерживает как последовательные модели (стек слоев), так и функциональный API для создания более сложных архитектур с несколькими входами, выходами или общими слоями.
*   **Обширный набор слоев и моделей:** Предоставляет готовые к использованию слои (полносвязные, сверточные, рекуррентные и т.д.), функции активации, оптимизаторы и функции потерь.
*   **Легкое прототипирование:** Упрощает процесс экспериментирования с различными архитектурами и гиперпараметрами.
*   **Интеграция с TensorFlow:** Будучи частью TensorFlow, Keras получает все преимущества базовой библиотеки, включая поддержку различных устройств и распределенных вычислений.

**Таким образом**, TensorFlow предоставляет мощную низкоуровневую инфраструктуру для численных вычислений и машинного обучения, а Keras предоставляет удобный высокоуровневый интерфейс для быстрого и эффективного построения и обучения нейронных сетей на базе TensorFlow. В этом практическом занятии мы будем использовать `tf.keras` для построения и обучения наших моделей.

### Архитектура модели

#### Создание модели: `tf.keras.models.Sequential`

`tf.keras.models.Sequential` - это самый простой способ создания моделей в Keras. Он позволяет создавать модель как последовательность слоев. Данные проходят через каждый слой по порядку. Это подходит для большинства стандартных задач, где нет необходимости в создании моделей со сложной топологией.

#### Слои модели: `tf.keras.layers.Dense`

`tf.keras.layers.Dense` представляет собой обычный полносвязный слой нейронной сети. Каждый нейрон в этом слое связан со всеми нейронами предыдущего слоя. Слой имеет следующие параметры:
*   **`units`**: Количество нейронов в слое. Это ключевой параметр, определяющий размерность выходного пространства слоя . Количество нейронов должно быть в несколько раз меньше количества обучающих примеров при условии избыточности обучающих данных. Если у нас будет слишком мало нейронов — сеть не обучится, ошибка при работе останется большой (ошибка обобщения).
Слишком много нейронов — сеть переобучится: выходной вектор будет передавать незначительные и несущественные детали в изучаемой зависимости, например, шум или ошибочные данные.
*   **`activation`**: Функция активации, применяемая к выходу каждого нейрона после взвешенной суммы входных сигналов и смещения. Нелинейные функции активации необходимы для того, чтобы модель могла обучаться сложным, нелинейным зависимостям в данных.
*   **`input_shape`**: Форма входных данных для *первого* слоя модели. TensorFlow может автоматически выводить форму входа для последующих слоев, но для первого слоя ее обычно необходимо указывать явно.

#### Функции активации, которые мы будем использовать сегодня

*   **`'relu'` (Rectified Linear Unit)**: `max(0, x)`. Одна из самых популярных функций для скрытых слоев благодаря своей простоте и эффективности.
*   **`'linear'`**: `f(x) = x`. Прямая функция активации, которая просто передает входное значение без изменений. Используется в выходном слое для задач регрессии, где нам нужно предсказать непрерывное значение без ограничения диапазона.
*   **`'sigmoid'`** (Сигмоида): Сжимает входное значение в диапазон от 0 до 1. Часто используется в выходных слоях для задач бинарной классификации, поскольку ее выход можно интерпретировать как вероятность.



In [ ]:
#  Задание архитектуры нейронной сети
model = Sequential()

# Добавление входного слоя и скрытого слоя с функцией активации ReLU
model.add(Dense(units=64, activation='relu', input_shape=(X_train.shape[1],)))

# Добавление еще одного скрытого слоя с функцией активации ReLU
model.add(Dense(units=32, activation='relu'))

# Добавление выходного слоя с линейной функцией активации для регрессии
model.add(Dense(units=1, activation='linear'))

# Краткое описание модели
model.summary()

Метод summary() выводит краткое описание структуры модели в табличной форме.

*   Столбец Layer (type) содержит информацию о типе слоя (пока что только полносвязные слои)
*   Столбец Output Shape показывает размерность выхода соответсвующего слоя (у нас совпадает с колличеством нейронов в полносвязном слое)
*   Столбец Param # содержит общее количество обучаемых параметров модели, относящихся к данному слою:

количество параметров = веса + смещения = размерность входных данных * количество нейронов + количество нейронов = (размерность входных данных + 1) * количество нейронов

params = (input shape + 1) * units number

Форма X_train: (800, 10) - 800 примеров для обучения, у каждого по 10 признаков

Примеры подаются на вход модели один за другим.

1) 704 = (10 + 1) * 64
2) 2080 = (64 + 1) * 32
3) 33 = (32 + 1) * 1

В конце выводится общее количество параметров (сумма всех весов и смещений), которые делятся на обучаемые и необучаемые. Если не ставить дополнительные условия, то все параметры модели будут обучаемыми (то есть изменяющимися на этапе обновления весов - backpropagtion). Необучаемые параметры можно использовать по необходимости, например, задав параметр слоя `trainable = False`, если мы не хотим менять значения параметров некоторых слоёв при дообучении готовой модели. Они также могут появиься при использовании BatchNormalization.


## Компиляция модели

Компиляция модели TensorFlow является важным шагом перед обучением. Она включает в себя настройку модели для обучения путем указания оптимизатора, функции потерь и метрик.

### [Оптимизатор](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers)

Оптимизатор - это алгоритм, используемый для минимизации функции потерь во время обучения путем настройки весов и смещений модели. К числу распространенных оптимизаторов в TensorFlow относятся следующие разновидности градиентного спуска:

- **Adam:** Адаптивный алгоритм оптимизации скорости обучения, который, как правило, эффективен на практике
- **SGD (стохастический градиентный спуск):** Базовый оптимизатор, который обновляет веса на основе градиента функции потерь в одном обучающем примере или небольшой партии
- **RMSProp (распространение среднеквадратичного значения):** Оптимизатор, который адаптирует скорость обучения для каждого параметра на основе среднеквадратичного значения последних градиентов

### Функция потерь

Функция потерь количественно определяет разницу между прогнозами модели и фактическими целевыми значениями. Цель обучения - минимизировать эти потери. К наиболее распространенным функциям потерь в TensorFlow относятся:

- **Среднеквадратичная ошибка (MSE):** Используется для задач регрессии, вычисляет среднее значение квадратов различий между прогнозируемыми и фактическими значениями
- **Категориальная кроссэнтропия:** Используется для задач классификации по нескольким классам, измеряет производительность классификационной модели, результатом которой является значение вероятности от 0 до 1.
- **Бинарная кроссэнтропия:** Используется для задач бинарной классификации, это частный случай категориальной кроссэнтропии для двух классов.

### Метрики

Метрики используются для оценки производительности модели во время обучения и тестирования. В отличие от функции потерь, которая может иметь специфическую форму для оптимизации, метрики часто более интуитивно понятны для человека. Общие показатели в TensorFlow включают:
- **Точность:** Используется для задач классификации, измеряет долю правильно классифицированных экземпляров.
- **Средняя абсолютная ошибка (MAE):** Используется для задач регрессии, рассчитывает среднее абсолютных разностей между предсказаниями и истинными значениями, предоставляя меру средней ошибки в тех же единицах, что и выходные данные.


#### Метод компиляции модели: `model.compile()`

Метод `model.compile()` конфигурирует модель для обучения. На этом этапе мы определяем:
*   **`optimizer`**: Алгоритм оптимизации, который будет использоваться для обновления весов модели в процессе обучения с целью минимизации функции потерь. Например, `'adam'` - популярный и эффективный оптимизатор.
*   **`loss`**: Функция потерь, которая измеряет разницу между предсказанными моделью значениями и истинными метками. Цель обучения - минимизировать эту функцию. Для нашей задачи регрессии мы использовали `'mse'` (Mean Squared Error - Среднеквадратическая ошибка).
*   **`metrics`**: Список метрик, которые будут вычисляться и отслеживаться в процессе обучения и оценки. Они используются для оценки производительности модели и не влияют на процесс оптимизации. Для регрессии часто используется `'mae'` (Mean Absolute Error - Средняя абсолютная ошибка) в дополнение к MSE.



In [ ]:
# Compile the model for the regression task
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

print("\nМодель успешно скомпилирована с оптимизатором Adam, функцией потерь Mean Squared Error loss, и метрикой Mean Absolute Error.")

### Тема 4. TensorFlow: обучение модели

### Обучение модели
Обучение нейронной сети включает в себя итеративную настройку весов и смещений модели для минимизации функции потерь на тренировочных данных.

#### Эпохи
Эпоха - это один полный проход по всему набору обучающих данных в процессе обучения. В течение одной эпохи модель видит каждый пример тренировочных данных один раз.

Обучение в течение нескольких эпох позволяет модели многократно извлекать важные признаки из данных, что потенциально повышает ее производительность.

#### Мини-батчи
Использование батчей (наборов тренировочных данных) помогает уменьшить шум при обновлении градиента по сравнению со SGD и является более эффективным в вычислительном отношении для больших наборов данных, чем обычный GD. Большие батчи могут давать более стабильные градиенты, но требуют больше памяти.

#### Валидационная выборка
Валидационная выборка - это часть тренировочных данных, которая откладывается и используется для оценки производительности модели во время обучения, но не используется для обновления весов модели.  Модель оценивается на этих данных после каждой эпохи, что помогает отслеживать переобучение, когда модель хорошо работает с тренировочными данными, но плохо работает с тестовыми.


#### Обучение модели: `model.fit()`

Метод `model.fit()` запускает процесс обучения модели. Параметры:
*   **`epochs`**: Количество полных проходов по всему тренировочному набору данных.
*   **`batch_size`**: Размер мини-батча. Определяет количество примеров, используемых в каждой итерации градиентного спуска для вычисления градиента и обновления весов.
*   **`validation_split`**: Доля данных из тренировочного набора, которая будет использоваться в качестве валидационного набора.


In [ ]:
# Обучите скомпилированную модель
history = model.fit(X_train, y_train,
                    epochs=100,          # Укажите количество эпох
                    batch_size=32,       # Установите размер батча
                    validation_split=0.2) # Добавьте долю валидационной выборки

## Оценка модели

### Оценка модели
После обучения важно оценить производительность модели на отдельном тестовом наборе данных, который модель не видела во время обучения. Это дает объективную оценку того, насколько хорошо модель будет обобщена на новые, ранее не виденные ей, данные.


#### Оценка модели: `model.evaluate()`

Метод `model.evaluate()` оценивает производительность обученной модели на предоставленных данных (обычно на тестовом наборе). Он возвращает значения функции потерь и метрик, указанных при компиляции модели.



In [ ]:
# Оценим модель на тестовом наборе данных
loss, mae = model.evaluate(X_test, y_test, verbose=0)

print(f"\nTest Loss (MSE): {loss:.4f}")
print(f"Test Mean Absolute Error (MAE): {mae:.4f}")

## Тема 5. TensorFlow: инференс (построение прогноза/вывод) модели

### Построение прогноза модели
Инференс - это процесс использования обученной модели машинного обучения для составления прогнозов на основе новых данных. Как только модель обучена и оценена, ее можно использовать для составления прогнозов на основе реальных данных.

Это заключительный шаг, на котором изученные шаблоны применяются моделью для решения исходной задачи.

#### Построение прогноза: `model.predict()`

Метод `model.predict()` используется для генерации прогноза для заданных входных данных, используя обученную модель. Он возвращает предсказанные моделью значения для каждого примера во входном наборе данных.


In [ ]:
# Инференс на тестовом наборе данных
predictions = model.predict(X_test)

print("\n### Предсказанные значения")
print("Первые несколько предсказаний модели на тестовом наборе:")
# Print the first 5 predictions
for i in range(5):
    print(f"Prediction {i+1}: {predictions[i][0]:.4f}")

# Задание 2: Модель классификации

Задайте архитектуру нейронной сети как последовательную модель с 2 скрытыми полносвязными слоями, в которых будет 128 и 64 нейрона с функцией активации ReLU. В выходном слое используйте функцию активации sigmoid (для бинарной классификации).

In [ ]:
# Генерируем данные для задачи классификации
X_cls, y_cls = make_classification(n_samples=1000, n_features=20, n_informative=10, n_redundant=5, n_classes=2, random_state=42)


X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)


print("Shape of X_cls_train:", X_cls_train.shape)
print("Shape of X_cls_test:", X_cls_test.shape)
print("Shape of y_cls_train:", y_cls_train.shape)
print("Shape of y_cls_test:", y_cls_test.shape)

In [ ]:
#### ВСТАВЬТЕ КОД СЮДА

model_cls = ...

####


model_cls.summary()

Задайте функцию компиляции модели с оптимизатором Adam, бинарной кросс-энтропией и метрикой accuracy (точность).


In [ ]:
#### ВСТАВЬТЕ КОД СЮДА

...

####

print("\nМодель классификации успешно скомпилирована.")

Обучите построенную модель классификации со следующими гиперпараметрами: количество эпох 100, размер мини-батча 32, доля валидационной выборки 0.2, verbose=1.



In [ ]:
# Обучить скомпилированную модель классификации
#### ВСТАВЬТЕ КОД СЮДА

history_cls = ...

####

Оцените обученнцю модель классификации



In [ ]:
# Оценить обученную модель классификации на тестовом наборе данных

#### ВСТАВЬТЕ КОД СЮДА

loss_cls, accuracy_cls = ...

####


print(f"\nФункция потерь (Binary Crossentropy): {loss_cls:.4f}")
print(f"Точность (Accuracy): {accuracy_cls:.4f}")

In [ ]:
# Сделать предсказания на тестовом наборе данных классификации

#### ВСТАВЬТЕ КОД СЮДА

predictions_cls = ...

####

print("Ниже приведены первые несколько предсказаний, сделанных обученной моделью на тестовом наборе данных.")
print("Для бинарной классификации вывод представляет собой предсказанную вероятность принадлежности к положительному классу (близко к 1) или отрицательному классу (близко к 0).")
# Вывести первые 5 предсказаний и соответствующие истинные метки
for i in range(5):
    # Для бинарной классификации sigmoid возвращает вероятность положительного класса
    predicted_prob = predictions_cls[i][0]
    # Можно также преобразовать вероятность в предсказанную метку класса (0 или 1)
    predicted_label = 1 if predicted_prob > 0.5 else 0
    print(f"Предсказание {i+1}: Вероятность положительного класса = {predicted_prob:.4f} | Предсказанная метка: {predicted_label} | Истинная метка: {y_cls_test[i]}")

### Пояснения, касающиеся выбра параметров архитектуры и обучения модели классификации

#### Функция активации выходного слоя: `'sigmoid'`

Для задачи бинарной классификации (когда у нас только два класса, например, 0 или 1) на выходном слое обычно используется функция активации `'sigmoid'`.
*   **Сигмоида (`sigmoid`):** Сжимает выходное значение нейрона в диапазон от 0 до 1. Это значение может интерпретироваться как вероятность принадлежности входного примера к положительному классу (обычно обозначаемому как 1). Если вероятность выше определенного порога (часто 0.5), пример классифицируется как положительный, иначе — как отрицательный.

Для многоклассовой классификации (более двух классов) на выходном слое используется функция активации `'softmax'`.
*   **Softmax:** Преобразует вектор входных значений в распределение вероятностей по всем классам. Сумма всех выходных вероятностей равна 1. Выходное значение для каждого класса представляет собой предсказанную вероятность принадлежности входного примера к этому классу.

#### Функция потерь: `'binary_crossentropy'`

Для задач классификации используются функции потерь, которые эффективно измеряют "расстояние" между предсказанными вероятностями классов и истинными метками.
*   **`'binary_crossentropy'` (Бинарная кросс-энтропия):** Это стандартная функция потерь для задач **бинарной классификации**. Она измеряет, насколько сильно предсказанные вероятности (выход сигмоиды) отличаются от истинных бинарных меток (0 или 1). Минимизация этой функции потерь приводит к тому, что модель уверенно предсказывает высокие вероятности для истинного класса и низкие для ложного.

Для многоклассовой классификации обычно используется `'categorical_crossentropy'` или `'sparse_categorical_crossentropy'`.

#### Метрика: `'accuracy'`

Метрики помогают оценить производительность модели классификации в понятных терминах.
*   **`'accuracy'` (Точность):** Наиболее распространенная метрика для задач классификации. Она рассчитывает долю правильно классифицированных примеров от общего числа примеров. Точность 1.0 означает, что модель правильно классифицировала все примеры, а точность 0.0 — ни одного.


### Основные выводы

*   Текущие модели дают базовые результаты. Дальнейшая настройка гиперпараметров (например, количество слоев, количество нейронов, скорость обучения, количество эпох) может потенциально улучшить результаты предсказаний моделей на тестовых наборах.
*   Визуализация истории обучения (функция потерь и метрики на обучающей и валидационной выборках по эпохам) для обеих моделей позволит лучше понять процесс обучения и выявить потенциальное переобучение (особенно для модели классификации, где наблюдается расхождение между точностью на обучении и валидации).
*   Для модели классификации, при наличии признаков переобучения, могут быть применены стратегии регуляризации (например, Dropout, L1/L2 регуляризация) для улучшения обобщающей способности. Мы подробнее рассмотрим их на следующих семинарах.

# Задание 3:
Добавьте графики метрик и функций потерь для визуализации процесса обучения (используйте переменные history_cls и history).

# Задание 4:
Поэкспериментируйте с архитектурами нейронных сетей на сайте https://playground.tensorflow.org/